# Predicting WRKY33 binding sites in *Arabidopsis thaliana*: can a Transformer improve on the standard PWM?

## The problem

Genes are switched on and off by **transcription factors (TFs)**, proteins that bind short DNA
sequences near genes. Knowing **where a TF binds** tells us which genes it can regulate.
**WRKY33** is a TF of the plant immune response. It binds the **W-box**, a short DNA motif whose
core is `TTGAC[C/T]`.

**The standard method** to predict TF binding sites is the **position weight matrix (PWM)**. A PWM
is a table of base preferences at each position of the motif; a DNA sequence is scored by how well
its best stretch matches that table. PWMs are simple, fast and easy to interpret. But a PWM only
looks at a short motif (here 10 bp) and ignores the sequence around it, and the W-box occurs many
thousands of times in the genome while WRKY33 binds only some of these copies.

**The challenger** is a **Transformer**, a DNA language model pretrained on many genomes, which we
fine-tune on WRKY33 binding data. It reads the whole 201-bp sequence, not only the motif.

**Question of the exercise:** can the Transformer predict WRKY33 binding **better than the
standard PWM**?

## Where the binding data come from: DAP-seq

The binding sites come from **DAP-seq** (O'Malley *et al.* 2016, *Cell*). In short: genomic DNA
of *Arabidopsis* is cut into fragments of about 200 bp and mixed with purified WRKY33 protein
attached to magnetic beads. The fragments bound by WRKY33 are kept, sequenced and mapped back to
the genome. Where many reads pile up there is a **peak**; its **summit** is the position of
strongest binding. Because DAP-seq uses purified DNA, it shows where WRKY33 *can* bind the
sequence, outside the living cell.

## What we do in this notebook

1. **Bound sequences** (positives): 201 bp around each WRKY33 peak summit.
2. **Unbound sequences** (negatives): random 201-bp regions of the genome that are not WRKY33 peaks.
3. **Split by chromosome:** train on chr1 to chr3, validate on chr4, test on chr5.
4. **Standard method:** score every sequence with the JASPAR WRKY33 PWM (no training needed).
5. **Challenger:** fine-tune the Transformer on the training chromosomes.
6. **Compare** both on chr5, and look at which sequences the Transformer gets right that the PWM
   does not.

| Section | Content |
|---|---|
| 1 | Setup |
| 2 | Download the data |
| 3 | Build the dataset |
| 4 | The standard method: the PWM |
| 5 | The challenger: the Transformer |
| 6 | Evaluation on chromosome 5 (with an explanation of ROC curves and AUC) |
| 7 | Can the Transformer improve on the PWM? |
| 8 | Summary |
| 9 | Structure of your report |

**Before you start:** choose `Runtime > Change runtime type > GPU`. Run the cells from top to
bottom, in order. If the runtime disconnects, start again from the first cell.

# 1. Setup

**Check the GPU.** `nvidia-smi` shows the GPU attached to this Colab session. The Transformer
needs a GPU; if you see an error, change the runtime type before continuing.

In [ ]:
!nvidia-smi

**Install the Python packages** that Colab does not have: **Biopython** (reads the genome file),
**logomaker** (draws the motif logo) and the HuggingFace libraries **transformers, datasets,
accelerate** (load and fine-tune the Transformer).

In [ ]:
!pip install -q biopython logomaker transformers datasets accelerate
print("python packages installed")

**Import the libraries.** The cell also fixes the random seed (`SEED`), so that the results are
the same when the notebook is re-run, and selects the GPU (`cuda`). It prints the device used.

In [ ]:
import gzip, json, re, time, types, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, logomaker
import torch
from numpy.lib.stride_tricks import sliding_window_view
from Bio import SeqIO
from sklearn.metrics import roc_auc_score, roc_curve

SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# 2. Download the data

The cell downloads into `/content/data`:
- the **DAP-seq peaks** (Plant Cistrome Database, about 78 MB archive), from which only the
  **WRKY33** peak file is extracted;
- the ***Arabidopsis* genome**, TAIR10 (about 36 MB) from Ensembl Plants.

Each download is skipped if the file is already there. The cell ends by listing the files.

In [ ]:
%%bash
set -e
mkdir -p /content/data && cd /content/data
[ -f peaks.zip ] || wget -q http://neomorph.salk.edu/dap_web/pages/dap_data_v4/fullset/dap_download_may2016_peaks.zip -O peaks.zip
unzip -qo peaks.zip 'dap_data_v4/peaks/WRKY_tnt/WRKY33_col_a/*'
[ -f tair10.fa.gz ] || wget -q https://ftp.ensemblgenomes.ebi.ac.uk/pub/plants/current/fasta/arabidopsis_thaliana/dna/Arabidopsis_thaliana.TAIR10.dna.toplevel.fa.gz -O tair10.fa.gz
ls -la /content/data

**Load the genome and the WRKY33 peaks.**
- The 5 chromosomes are read into memory (`GENOME`).
- The peak file has one line per peak: chromosome, start, ..., and in the last column the
  position of the **summit** relative to the start.
- For every peak we cut the summit plus or minus 100 bp (`HALF`) from the genome, a **201-bp**
  sequence (`WIN`), and drop sequences containing `N` (unknown bases).

The cell prints the chromosome sizes and the number of WRKY33 peaks per chromosome.

In [ ]:
PEAK_FILE = "/content/data/dap_data_v4/peaks/WRKY_tnt/WRKY33_col_a/chr1-5/chr1-5_GEM_events.narrowPeak"

with gzip.open("/content/data/tair10.fa.gz", "rt") as f:      # Ensembl names chromosomes 1..5
    GENOME = {f"chr{r.id}": str(r.seq).upper() for r in SeqIO.parse(f, "fasta")
              if r.id in {"1", "2", "3", "4", "5"}}
print({c: f"{len(s)/1e6:.1f} Mb" for c, s in GENOME.items()})

HALF = 100                       # window = summit +/- 100 bp
WIN = 2 * HALF + 1               # 201 bp
ACGT_ONLY = re.compile(f"[ACGT]{{{WIN}}}")

peaks = pd.read_csv(PEAK_FILE, sep="\t", header=None)
pos = pd.DataFrame({"chrom": peaks[0], "summit": peaks[1] + peaks[9]})     # start + summit offset
pos["seq"] = [GENOME[c][s - HALF:s + HALF + 1] for c, s in zip(pos.chrom, pos.summit)]
pos = pos[pos.seq.map(lambda s: bool(ACGT_ONLY.fullmatch(s)))].reset_index(drop=True)
print(f"WRKY33: {len(pos):,} peaks |", pos.chrom.value_counts().sort_index().to_dict())

# 3. Build the dataset

**Positives** (label 1) are the WRKY33 peak sequences. **Negatives** (label 0) are random 201-bp
regions of the genome (`random_genomic`), taken from the same chromosomes in the same numbers as
the positives, that do not overlap any WRKY33 peak and contain no `N`.

**Split by chromosome.** Whole chromosomes are kept apart, so no test sequence comes from the same
region as a training sequence:

| split | chromosomes | used for |
|---|---|---|
| train | chr1, chr2, chr3 | fine-tuning the Transformer |
| val | chr4 | choosing the Transformer's best epoch and the decision thresholds of both methods |
| test | chr5 | final comparison only |

The cell prints the number of positive (1) and negative (0) sequences in each split.

In [ ]:
rng = np.random.default_rng(SEED)

def random_genomic(n, chrom):
    """n random N-free windows on `chrom` that do not overlap any WRKY33 peak window"""
    taken = np.sort(pos.loc[pos.chrom == chrom, "summit"].values)
    g, out = GENOME[chrom], []
    while len(out) < n:
        s = int(rng.integers(HALF, len(g) - HALF))
        i = np.searchsorted(taken, s)
        if (i < len(taken) and taken[i] - s < WIN) or (i > 0 and s - taken[i - 1] < WIN):
            continue
        seq = g[s - HALF:s + HALF + 1]
        if ACGT_ONLY.fullmatch(seq):
            out.append((chrom, s, seq))
    return pd.DataFrame(out, columns=["chrom", "summit", "seq"])

neg = pd.concat([random_genomic(n, c) for c, n in pos.chrom.value_counts().items()])

SPLIT = {"chr1": "train", "chr2": "train", "chr3": "train", "chr4": "val", "chr5": "test"}
data = pd.concat([pos.assign(label=1), neg.assign(label=0)], ignore_index=True)
data["split"] = data.chrom.map(SPLIT)
data = data.sample(frac=1, random_state=SEED).reset_index(drop=True)
train, val, test = (data[data.split == s] for s in ("train", "val", "test"))
print(data.groupby(["split", "label"]).size().unstack())

**Composition check.** The histogram compares the GC content (fraction of G and C bases) of the
bound and the unbound sequences, and the cell prints the mean of each group.

Hint: if bound and unbound sequences differ in GC content, a trained model could separate them
partly by composition, without using the binding motif. The PWM cannot do this.

In [ ]:
gc = lambda s: (s.count("G") + s.count("C")) / len(s)
fig, ax = plt.subplots(figsize=(7, 3.5))
for lab, name in [(1, "bound (WRKY33 peaks)"), (0, "unbound (random genome)")]:
    ax.hist(data[data.label == lab].seq.map(gc), bins=40, alpha=.5, density=True, label=name)
ax.set_xlabel("GC content"); ax.set_ylabel("density"); ax.legend(); plt.tight_layout(); plt.show()
print(data.groupby("label").seq.apply(lambda x: x.map(gc).mean()).rename({0: "unbound", 1: "bound"}).round(3))

# 4. The standard method: the JASPAR PWM

The cell downloads the WRKY33 matrix `MA1301.2` from the JASPAR database: for each of the 10 motif
positions, how often each base was observed. It prints the matrix ID, name, species and the type
of data it was derived from. The counts are then turned into a **PWM** of log-odds scores:

$$\text{PWM}_{b,i} = \log_2 \frac{p_{b,i}}{q_b}$$

where $p_{b,i}$ is the frequency of base $b$ at motif position $i$ (with a small pseudocount of
0.25 per base) and $q_b$ is the frequency of base $b$ in the training chromosomes (the
background). A positive value means the base is preferred at that position, a negative value that
it is avoided. The cell prints the background frequencies and draws the motif as a **sequence
logo**: the taller a letter, the more important that base is at that position.

The PWM is **not trained**: only its decision threshold will be chosen on the validation set
(Section 6).

In [ ]:
jaspar = json.load(urllib.request.urlopen("https://jaspar.elixir.no/api/v1/matrix/MA1301.2/?format=json"))
print(jaspar["matrix_id"], jaspar["name"], "| species:", jaspar["species"][0]["name"],
      "| derived from:", jaspar["type"])

PFM = np.array([jaspar["pfm"][b] for b in "ACGT"], dtype=float)          # 4 x L counts
PPM = (PFM + 0.25) / (PFM.sum(0) + 1)                                     # probabilities
train_genome = "".join(GENOME[c] for c, s in SPLIT.items() if s == "train")
BG = np.array([train_genome.count(b) for b in "ACGT"], dtype=float); BG /= BG.sum()
PWM = np.log2(PPM / BG[:, None])
MOTIF_LEN = PWM.shape[1]
print("background ACGT:", BG.round(3), "| motif length:", MOTIF_LEN)

logo = pd.DataFrame(PPM.T, columns=list("ACGT"))
fig, ax = plt.subplots(figsize=(5, 2))
logomaker.Logo(logomaker.transform_matrix(logo, from_type="probability", to_type="information"), ax=ax)
ax.set_title("JASPAR MA1301.2 (WRKY33)"); ax.set_ylabel("bits"); plt.show()

**Scoring sequences with the PWM.**
- `one_hot` turns sequences into 0/1 matrices (one column per base A, C, G, T).
- `revcomp` gives the reverse complement of a sequence (the other DNA strand).
- `motif_score` slides the PWM along each 201-bp sequence, adds up the PWM values of the bases at
  every position, does the same with the reverse complement of the PWM (the TF can bind either
  strand), and keeps the **best** score. Only the best-matching 10-bp stretch counts.

The cell prints the PWM score of the first bound and the first unbound test sequence as an
example.

In [ ]:
LUT = np.zeros((256, 4), dtype=np.float32)
for i, b in enumerate(b"ACGT"):
    LUT[b, i] = 1

def one_hot(seqs):
    """list of equal-length sequences -> N x L x 4 array"""
    return LUT[np.frombuffer("".join(seqs).encode(), dtype=np.uint8)].reshape(len(seqs), -1, 4)

def revcomp(s):
    return s[::-1].translate(str.maketrans("ACGT", "TGCA"))

def motif_score(seqs):
    """best PWM match per sequence, over all positions and both strands"""
    W = sliding_window_view(one_hot(seqs), MOTIF_LEN, axis=1)       # N x positions x 4 x L
    fwd = np.einsum("npbl,bl->np", W, PWM)
    rev = np.einsum("npbl,bl->np", W, PWM[::-1, ::-1])             # reverse complement of the PWM
    return np.maximum(fwd, rev).max(1)

example = [test[test.label == 1].seq.iloc[0], test[test.label == 0].seq.iloc[0]]
print("PWM score, bound example:   %.2f" % motif_score(example)[0])
print("PWM score, unbound example: %.2f" % motif_score(example)[1])

# 5. The challenger: a DNA Transformer, fine-tuned

### The model
`NT_MODEL = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"` (Nucleotide Transformer v2,
about 50 million parameters) was pretrained on the genomes of many species and has learned general
features of DNA, but knows nothing about WRKY33. It cuts DNA into **6-letter words (6-mers)**, so
a 201-bp sequence becomes 37 tokens (33 six-mers, 3 single bases and a start token). Unlike the
PWM, it reads the whole 201-bp sequence at once.

### How we train it (fine-tuning)
1. A new output layer with **2 outputs** (unbound, bound) is added on top of the pretrained model.
2. **All weights** are then trained on the training sequences (chr1 to chr3). Each sequence is
   given twice, as it is and as its reverse complement, because the TF can bind either strand.
3. Settings: learning rate 3e-5, batch size 32, up to 8 epochs (passes over the training data).
4. After every epoch the model is checked on the validation sequences (chr4), and the **best
   epoch** (highest validation ROC-AUC) is kept.
5. To predict a sequence, the model scores both strands and the two probabilities are averaged.

### Load the model
The first part of the next cell contains **compatibility patches**: the model code stored online
was written for an older version of the `transformers` library, and the patches add back helpers
that newer versions removed. They do not change the model.

The second part loads the pretrained model with its new 2-output layer. The load report it prints
lists `lm_head.* UNEXPECTED` (the part used in pretraining, which is dropped) and
`classifier.* MISSING` (the new output layer, which will be trained). This is expected. Finally,
the cell checks that a 201-bp sequence fits in `NT_MAXLEN` tokens and prints its token count.

In [ ]:
import transformers
import transformers.pytorch_utils as pt_utils
from transformers import (AutoTokenizer, AutoConfig, AutoModelForSequenceClassification,
                          PreTrainedModel, TrainingArguments, Trainer)
from datasets import Dataset

# --- compatibility patches for the model code on the HuggingFace Hub (written for older transformers) ---
PreTrainedModel.all_tied_weights_keys = {}
if not hasattr(pt_utils, "find_pruneable_heads_and_indices"):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        if len(heads) == 0:
            return heads, None
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            mask[head] = 0
        mask = mask.view(-1).eq(1)
        return heads, torch.arange(len(mask))[mask].long()
    pt_utils.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices

def _get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
    if attention_mask.dim() == 3:
        ext = attention_mask[:, None, :, :]
    elif attention_mask.dim() == 2:
        ext = attention_mask[:, None, None, :]
    else:
        raise ValueError("Wrong shape for attention_mask")
    ext = ext.to(dtype=torch.float32)
    return (1.0 - ext) * torch.finfo(torch.float32).min

def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
    return head_mask if head_mask is not None else [None] * num_hidden_layers

# --- load the pretrained model with a new 2-output layer ---
NT_MODEL = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
tok = AutoTokenizer.from_pretrained(NT_MODEL, trust_remote_code=True)
cfg = AutoConfig.from_pretrained(NT_MODEL, trust_remote_code=True)
cfg.num_labels = 2
cfg.id2label, cfg.label2id = {0: "unbound", 1: "bound"}, {"unbound": 0, "bound": 1}
cfg.add_cross_attention, cfg.is_decoder = False, False
model = AutoModelForSequenceClassification.from_pretrained(
    NT_MODEL, config=cfg, trust_remote_code=True, low_cpu_mem_usage=False).to(device)
for mod in [model] + ([model.esm] if hasattr(model, "esm") else []):
    if not hasattr(mod, "get_extended_attention_mask"):
        mod.get_extended_attention_mask = types.MethodType(_get_extended_attention_mask, mod)
    if not hasattr(mod, "get_head_mask"):
        mod.get_head_mask = types.MethodType(_get_head_mask, mod)

NT_MAXLEN = 48        # max tokens; a 201-bp sequence is 37 tokens -> no truncation; padding is masked
n_tok = len(tok(pos.seq.iloc[0])["input_ids"])
assert n_tok < NT_MAXLEN, "sequences would be truncated: raise NT_MAXLEN"
print("tokens for one", WIN, "bp sequence:", n_tok)

### Fine-tune the Transformer
- `to_hf` converts sequences into tokens (padded to `NT_MAXLEN`; the padding is ignored by the
  model) and labels into a HuggingFace dataset.
- `nt_metrics` computes the validation ROC-AUC after each epoch (ROC-AUC is explained in
  Section 6).
- The HuggingFace `Trainer` fine-tunes the model with the settings above. During training it shows
  a table with, for each epoch, the training loss, the validation loss and the validation ROC-AUC.
  At the end it reloads the best epoch.
- `nt_predict` gives the probability of binding for any list of sequences, averaged over both
  strands.

This is the slowest cell of the notebook. It prints the training time.

In [ ]:
def tok_fn(b):
    enc = tok(b["seq"], truncation=True, padding="max_length", max_length=NT_MAXLEN)
    enc.pop("token_type_ids", None)
    return enc

def to_hf(seqs, labels=None):
    d = {"seq": list(seqs)}
    if labels is not None:
        d["label"] = list(labels)
    return Dataset.from_dict(d).map(tok_fn, batched=True, remove_columns=["seq"])

def nt_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    prob = torch.softmax(torch.tensor(logits).float(), -1)[:, 1].numpy()
    return {"auroc": roc_auc_score(p.label_ids, prob)}

train_ds = to_hf(list(train.seq) + [revcomp(s) for s in train.seq], list(train.label) * 2)
val_ds = to_hf(val.seq, val.label)
args = TrainingArguments(
    output_dir="/content/nt_wrky33",
    per_device_train_batch_size=32, per_device_eval_batch_size=128,
    num_train_epochs=8, learning_rate=3e-5,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="auroc", greater_is_better=True,
    logging_strategy="epoch", report_to="none", fp16=False, seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, compute_metrics=nt_metrics)
t0 = time.time(); trainer.train()
print(f"training time: {(time.time() - t0)/60:.1f} min")

def nt_predict(seqs):
    """probability of binding, averaged over both strands"""
    probs = []
    for s in (list(seqs), [revcomp(x) for x in seqs]):
        lg = trainer.predict(to_hf(s)).predictions
        lg = lg[0] if isinstance(lg, tuple) else lg
        probs.append(torch.softmax(torch.tensor(lg).float(), -1)[:, 1].numpy())
    return (probs[0] + probs[1]) / 2

**Learning curve.** Training and validation loss per epoch (left axis; lower is better), the
validation ROC-AUC (green, right axis; higher is better), and the best epoch (dotted line), whose
weights were kept.

Hint: where is the best epoch, and what happens to the validation loss afterwards? Note the
training time printed above: the PWM needed no training at all.

In [ ]:
h = pd.DataFrame(trainer.state.log_history)
tr_l, ev = h.dropna(subset=["loss"]), h.dropna(subset=["eval_loss"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tr_l.epoch, tr_l.loss, "-o", ms=3, label="train loss")
ax.plot(ev.epoch, ev.eval_loss, "-o", ms=3, label="val loss")
ax.axvline(ev.epoch.iloc[ev.eval_auroc.values.argmax()], color="grey", ls=":", label="best epoch")
ax2 = ax.twinx(); ax2.plot(ev.epoch, ev.eval_auroc, "g--o", ms=3, label="val ROC-AUC"); ax2.set_ylabel("val ROC-AUC")
ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("Transformer fine-tuning")
ax.legend(ax.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0],
          ax.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1], loc="center right")
plt.tight_layout(); plt.show()

# 6. Evaluation on the held-out chromosome 5

Before we look at the results, this section explains the two tools we use to compare the methods:
the **ROC curve** and the **AUC**. Read it first; the next cell runs a small example so you can see
how they work before applying them to the real data.

### 6.1 What is a ROC curve? A step-by-step explanation

**Step 1. Each method gives a score, not a yes / no answer.**
For every test sequence, the PWM gives its best-match score and the Transformer gives a probability
of binding. A higher score means "more likely bound". You can think of each method as sorting all
the test sequences into a list, from "most likely bound" at the top to "least likely bound" at the
bottom. A good method puts the truly bound sequences near the top of its list.

**Step 2. To get a yes / no answer we need a threshold.**
We choose a cut-off value: every sequence scoring at or above it is **called bound**, every
sequence below it is **called unbound**. Comparing these calls with the truth gives four outcomes:

| | truly bound | truly unbound |
|---|---|---|
| **called bound** | true positive (TP): a correct find | false positive (FP): a false alarm |
| **called unbound** | false negative (FN): a miss | true negative (TN): a correct rejection |

From these we compute two rates:
- **True positive rate (TPR)** = TP / (TP + FN): of all truly bound sequences, the fraction we
  found. Think of it as the **hit rate**. Higher is better. (Also called recall or sensitivity.)
- **False positive rate (FPR)** = FP / (FP + TN): of all truly unbound sequences, the fraction we
  wrongly called bound. Think of it as the **false alarm rate**. Lower is better.

**Step 3. Moving the threshold is a trade-off.**
- A **low** threshold calls almost everything bound: we find nearly all bound sequences (high hit
  rate) but also raise many false alarms (high FPR).
- A **high** threshold calls very few sequences bound: few false alarms, but we miss many bound
  sequences.

So a single threshold gives only one snapshot, and the result depends on where we put it.

**Step 4. The ROC curve shows every threshold at once.**
We slide the threshold from the highest score down to the lowest, and for each threshold we plot
one point: FPR on the x-axis (false alarms) and TPR on the y-axis (hits). Joining the points gives
the **ROC curve** (receiver operating characteristic, a name inherited from radar engineering).
- It always starts at (0, 0): threshold above every score, nothing called bound.
- It always ends at (1, 1): threshold below every score, everything called bound.
- **A perfect method** finds all bound sequences before raising any false alarm: its curve goes
  straight up to the top-left corner (0, 1), then across.
- **A method that guesses at random** finds hits and false alarms at the same rate: its curve
  follows the **diagonal**.
- **The closer the curve bends towards the top-left corner, the better the method.**

**Step 5. The AUC summarises the curve in one number.**
**ROC-AUC** is the **area under the ROC curve**. It runs from 0 to 1:

| ROC-AUC | meaning |
|---|---|
| 1.0 | perfect: every bound sequence scores above every unbound one |
| about 0.9 or more | excellent |
| about 0.8 | good |
| about 0.7 | fair |
| 0.5 | no better than random guessing (the diagonal) |

(These labels are only a rough guide.) The AUC has a simple meaning: **if you pick one bound and
one unbound sequence at random, the AUC is the probability that the method gives the bound one the
higher score.** An AUC of 0.90 means the method ranks such a pair correctly 90% of the time.

**Why we use ROC-AUC to compare the PWM and the Transformer.**
The PWM gives log2-odds scores (for example from -10 to +12) and the Transformer gives
probabilities (from 0 to 1), so their raw scores cannot be compared. The ROC curve and the AUC only
depend on how each method **orders** the sequences, not on the scale of the scores, and they do
not need a threshold. This makes ROC-AUC a fair, single number for comparing the two methods.

### A small worked example
Imagine only 8 test sequences, 4 truly bound and 4 truly unbound, with these scores:

| | scores |
|---|---|
| bound | 0.9, 0.8, 0.6, 0.3 |
| unbound | 0.7, 0.4, 0.2, 0.1 |

- **One threshold, one point.** At threshold 0.5, the sequences called bound are 0.9, 0.8, 0.6
  (bound) and 0.7 (unbound). So TP = 3, FN = 1, FP = 1, TN = 3, which gives TPR = 3/4 = 0.75 and
  FPR = 1/4 = 0.25: the point (0.25, 0.75) on the ROC curve.
- **All thresholds, the whole curve.** Repeating this for every possible threshold gives the full
  curve (next cell).
- **The AUC by counting pairs.** There are 4 x 4 = 16 bound / unbound pairs. The bound sequence
  has the higher score in 13 of them (only 0.6 < 0.7, 0.3 < 0.7 and 0.3 < 0.4 are wrong), so
  ROC-AUC = 13 / 16 = 0.81. The area under the curve in the next cell is exactly the same number.


**Run the small example.** The cell tries every score of the 8 toy sequences as the threshold
and prints a table with, for each threshold, the true positives (TP), false positives (FP), TPR and
FPR. Each row is one point of the ROC curve. It then draws two plots:
- **left:** the 8 scores on a line, with the threshold 0.5; the dots to the right of the line are
  called bound;
- **right:** the ROC curve (blue staircase) built from the table, the area under it (shaded, the
  ROC-AUC), the diagonal of a random method, the path of a perfect method, and the red point for
  threshold 0.5.

Hint: follow the table row by row. Each time the threshold passes a **bound** sequence the curve
steps **up** (one more hit); each time it passes an **unbound** sequence it steps **right** (one
more false alarm). A good method makes all its up-steps first.


In [ ]:
# Toy example: 8 sequences (1 = bound, 0 = unbound) and their scores
toy_y = np.array([1, 1, 1, 1, 0, 0, 0, 0])
toy_s = np.array([0.9, 0.8, 0.6, 0.3, 0.7, 0.4, 0.2, 0.1])

# try every score as the threshold, from the highest to the lowest
rows = []
for t in sorted(toy_s, reverse=True):
    called = toy_s >= t
    tp, fp = (called & (toy_y == 1)).sum(), (called & (toy_y == 0)).sum()
    rows.append({"threshold": t, "TP": tp, "FP": fp,
                 "TPR (hits)": tp / 4, "FPR (false alarms)": fp / 4})
display(pd.DataFrame(rows))

fpr, tpr, _ = roc_curve(toy_y, toy_s)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# left: the scores on a line, with the threshold 0.5
ax = axes[0]
ax.scatter(toy_s[toy_y == 1], [1] * 4, s=120, color="tab:blue", label="truly bound")
ax.scatter(toy_s[toy_y == 0], [0] * 4, s=120, color="tab:orange", label="truly unbound")
for s_, l_ in zip(toy_s, toy_y):
    ax.annotate(f"{s_:.1f}", (s_, l_), textcoords="offset points", xytext=(0, 10), ha="center")
ax.axvline(0.5, color="grey", ls="--")
ax.text(0.52, 0.5, "threshold 0.5:\nright of the line\n= called bound", va="center")
ax.set_yticks([0, 1]); ax.set_yticklabels(["unbound", "bound"]); ax.set_ylim(-0.6, 1.6)
ax.set_xlabel("score"); ax.set_title("Scores of the 8 sequences"); ax.legend(loc="upper left")

# right: the ROC curve, the area under it, and the point for threshold 0.5
ax = axes[1]
ax.fill_between(fpr, tpr, step="post", alpha=0.2, label=f"area = ROC-AUC = {roc_auc_score(toy_y, toy_s):.2f}")
ax.step(fpr, tpr, where="post", lw=2, label="ROC curve")
ax.plot([0, 1], [0, 1], "k:", label="random guessing (AUC 0.5)")
ax.plot([0, 0, 1], [0, 1, 1], color="green", ls="--", label="perfect method (AUC 1.0)")
ax.plot(0.25, 0.75, "o", color="red", ms=9, label="threshold 0.5 (FPR 0.25, TPR 0.75)")
ax.set_xlabel("false positive rate (false alarms)"); ax.set_ylabel("true positive rate (hits)")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02); ax.set_aspect("equal")
ax.set_title("ROC curve of the toy example"); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()


### 6.2 Compute the metrics on the real test set

Now we apply these ideas to the real data.

Both methods score the **same test sequences** (chr5). For each method the cell computes:
- **ROC-AUC** from the scores (no threshold needed): the main comparison;
- **accuracy, TPR and FPR** at one threshold. The threshold is **chosen on the validation set**
  (`best_threshold`: the one with the highest validation accuracy) and then applied unchanged to
  the test set, so the test set is never used to tune anything. TPR and FPR are the two axes of
  the ROC curve, so this threshold is one point on each method's curve (shown in the next plot).

The results are shown in one table (`RESULTS`). The test scores (`SCORES`) and the bound /
unbound calls (`CALLS`) are kept for the next cells.

In [ ]:
METHODS = {"PWM (JASPAR)": motif_score, "Transformer": nt_predict}

def best_threshold(y, s):
    cand = np.unique(s)
    if len(cand) > 2000:
        cand = np.quantile(s, np.linspace(0, 1, 2001))
    return cand[np.argmax([((s >= c) == y).mean() for c in cand])]

y = test.label.values
SCORES, CALLS, rows = {}, {}, []
for method, f in METHODS.items():
    thr = best_threshold(val.label.values, f(val.seq.tolist()))
    s = f(test.seq.tolist())
    pred = s >= thr
    SCORES[method], CALLS[method] = s, pred
    rows.append({"method": method, "ROC-AUC": roc_auc_score(y, s), "accuracy": (pred == y).mean(),
                 "TPR (recall)": pred[y == 1].mean(), "FPR": pred[y == 0].mean()})
RESULTS = pd.DataFrame(rows).set_index("method").round(3)
RESULTS

### 6.3 The ROC curves of the two methods

**ROC curves** on the test set, one line per method, with the ROC-AUC in the legend. The dot on
each curve is the threshold chosen on the validation set (its FPR and TPR from the table above).
- **Left, full curve:** the dotted diagonal is a random method. The closer a curve gets to the
  top-left corner, the better.
- **Right, zoom on the top-left corner:** when both methods perform well, the curves are close to
  each other; the zoom shows where they differ. Where the two curves cross, each method is better
  in a different range of thresholds.

Hint: which curve is closer to the top-left corner? At the chosen thresholds, which method finds
more bound sequences (TPR), and which makes fewer false positives (FPR)?
Compare these curves with the toy example above: the real curves have many small steps
(thousands of test sequences), but they are read in exactly the same way.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, (xmax, ymin, title) in zip(axes, [(1, 0, "ROC curve (test, chr5)"),
                                          (0.3, 0.7, "Zoom on the top-left corner")]):
    for method, s in SCORES.items():
        fpr, tpr, _ = roc_curve(y, s)
        line, = ax.plot(fpr, tpr, label=f"{method} (ROC-AUC {roc_auc_score(y, s):.3f})")
        ax.plot(RESULTS.loc[method, "FPR"], RESULTS.loc[method, "TPR (recall)"], "o", ms=8,
                color=line.get_color(), label=f"{method}, chosen threshold")
    ax.plot([0, 1], [0, 1], "k:", lw=1, label="random (ROC-AUC 0.5)")
    ax.set(title=title, xlabel="false positive rate (FPR)", ylabel="true positive rate (TPR)",
           xlim=(0, xmax), ylim=(ymin, 1.005))
axes[0].legend(loc="lower right")
plt.tight_layout(); plt.show()

# 7. Can the Transformer improve on the PWM?

### 7a. Gain over the standard, and which sequences each method gets right
The first table gives the Transformer's value **minus** the PWM's value for each metric: for ROC-AUC,
accuracy and TPR, positive numbers mean the Transformer is better; for FPR it is the opposite,
negative numbers mean the Transformer makes fewer false positives (better). Since ROC-AUC runs from
0.5 (random) to 1 (perfect), judge a gain against the room left above the PWM: going from 0.90 to
0.95 removes half of the PWM's remaining ranking errors.

The second table uses the calls at the chosen thresholds. Separately for the **bound** test
sequences (true positives) and the **unbound** ones (false positives), it counts the sequences
called bound by **both** methods, by the **PWM only**, by the **Transformer only**, or by
**neither**.

Hint: an improvement can come from finding bound sequences that the PWM misses, from making fewer
false positives, or both. Which is it here?

In [ ]:
gain = (RESULTS.loc["Transformer"] - RESULTS.loc["PWM (JASPAR)"]).round(3)
display(gain.to_frame("Transformer minus PWM"))

pwm, nt = CALLS["PWM (JASPAR)"], CALLS["Transformer"]
rows = []
for lab, what in [(1, "bound, called bound (true positives)"), (0, "unbound, called bound (false positives)")]:
    m = y == lab
    rows.append({"test sequences": what, "n": m.sum(), "both": (pwm & nt & m).sum(),
                 "PWM only": (pwm & ~nt & m).sum(), "Transformer only": (~pwm & nt & m).sum(),
                 "neither": (~pwm & ~nt & m).sum()})
pd.DataFrame(rows).set_index("test sequences")

### 7b. Telling bound from unbound W-boxes
The W-box is short, so it also occurs in many unbound regions. The test sequences are split into
four groups: bound or unbound, with or without a W-box (`TTGAC[CT]` on either strand). For each
group the table gives the number of sequences and the percentage that each method **called
bound**. For bound sequences this percentage is the recall (higher is better); for unbound
sequences it is the false positive rate (lower is better).

Hint: how common is the W-box among bound and among unbound sequences? The hardest group for the
PWM is "unbound, with a W-box". Does the Transformer call fewer of them bound, while still
finding the bound ones?

In [ ]:
has_wbox = test.seq.str.contains(r"TTGAC[CT]|[AG]GTCAA").values      # W-box on either strand
rows = []
for lab, name in [(1, "bound"), (0, "unbound")]:
    for wb, wname in [(True, "with a W-box"), (False, "without a W-box")]:
        sel = (y == lab) & (has_wbox == wb)
        rows.append({"test sequences": f"{name}, {wname}", "n": sel.sum(),
                     **{f"% called bound, {m}": 100 * CALLS[m][sel].mean() if sel.any() else np.nan
                        for m in METHODS}})
pd.DataFrame(rows).set_index("test sequences").round(1)

# 8. Summary

In this notebook you:
- extracted 201-bp sequences around the WRKY33 DAP-seq peak summits (bound) and random genome
  regions (unbound), and split them by chromosome (train chr1 to chr3, validation chr4, test chr5);
- scored all sequences with the standard method, the JASPAR WRKY33 PWM;
- fine-tuned a pretrained DNA Transformer (Nucleotide Transformer v2, 50M) to tell bound from
  unbound sequences;
- compared both on chr5 with ROC curves and ROC-AUC, and with accuracy, TPR and FPR at a
  threshold chosen on the validation set;
- looked at where the Transformer gains or loses against the PWM.

# 9. Your report: structure and content

Write a short report (about 2 pages, figures and tables can be included and do not count as pages)

### 1. Introduction
- Transcription factors, WRKY33 and its role in plant immunity, and the W-box.
- What DAP-seq measures.
- The PWM as the standard method: how it works, its strengths and its limits.
- The Transformer as a possible improvement, in your own words.
- The aim: can the Transformer improve on the PWM?

### 2. Methodology
- **Data:** peak file, genome, window size, number of peaks.
- **Dataset:** how the bound and unbound sequences were made; the chromosome split.
- **PWM:** the JASPAR matrix `MA1301.2`, the background, scoring on both strands.
- **Transformer:** the model and how it was fine-tuned (settings, reverse complements, best epoch).
- **Evaluation:** explain in your own words the ROC curve and ROC-AUC (what is on each axis, what
  0.5 and 1 mean, why no threshold is needed), plus accuracy, TPR and FPR and how the thresholds
  were chosen.

### 3. Results
YOU CAN USE THESE STEP AS REFERENCE
1. **Composition check** (Section 3). *Do bound and unbound sequences differ in GC content?*
2. **Transformer training** (Section 5): the learning curve and the training time. *When was the
   best epoch?*
3. **PWM vs Transformer** (Section 6): the results table and the ROC curves. *Which method
   is better, and by how much?*
4. **Where the Transformer gains or loses** (Section 7): the gain table, the agreement table, and
   the W-box table. *Does the Transformer find bound sites the PWM misses? Does it make fewer or
   more false positives? Can it tell bound from unbound W-boxes better than the PWM?*

### 4. Discussion

USE THESE STEPS AS REFERENCE
1. **Did the Transformer improve on the standard?** Is the difference large enough to matter in
   practice?
2. **Where does an improvement (or a loss) come from?** Unbound regions that contain a W-box, the
   sequence around the motif, GC content, or something else? Use Sections 3 and 7.
3. **A fair comparison?** Recall where the JASPAR matrix comes from (printed in Section 4). The
   Transformer was trained on data from the same experiment it is tested on. How does this affect
   the comparison?
4. **Cost and interpretability.** The PWM needs no training and no GPU, and can be read directly
   from its logo. Is the Transformer worth its extra cost?
5. **Limitations** of this experiment (one TF, one DAP-seq experiment, random genomic negatives,
   window size, ...).

### 5. Conclusion
A short paragraph that answers the question from the Introduction: should the PWM remain the
standard for finding WRKY33 binding sites, or does the Transformer improve on it enough to replace
it? Base your answer on your results.